# **Mounting the drive**

In [23]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# **Loads standard libraries we’ll use throughout:**

**os for file paths and saving files**

**pandas for data tables**

**numpy for math/stat functions**

**Sets the folder location (BASE_PATH) where the Lahman dataset CSV files live in Drive.**

In [24]:
import os
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

BASE_PATH = "/content/drive/MyDrive/FADS/lahman_1871-2025_csv"

# **Dataset summary**

In [25]:

import pandas as pd

batting = pd.read_csv(f"{BASE_PATH}/Batting.csv")
pitching = pd.read_csv(f"{BASE_PATH}/Pitching.csv")
salaries = pd.read_csv(f"{BASE_PATH}/Salaries.csv")
people = pd.read_csv(f"{BASE_PATH}/People.csv")

print("Batting shape:", batting.shape)
print("Pitching shape:", pitching.shape)
print("Salaries shape:", salaries.shape)
print("People shape:", people.shape)

print("\nBatting columns:")
print(batting.columns.tolist())

print("\nPitching columns:")
print(pitching.columns.tolist())

print("\nSalaries columns:")
print(salaries.columns.tolist())

print("\nPeople columns:")
print(people.columns.tolist())

Batting shape: (128598, 22)
Pitching shape: (57630, 30)
Salaries shape: (26428, 5)
People shape: (24270, 25)

Batting columns:
['playerID', 'yearID', 'stint', 'teamID', 'lgID', 'G', 'AB', 'R', 'H', '2B', '3B', 'HR', 'RBI', 'SB', 'CS', 'BB', 'SO', 'IBB', 'HBP', 'SH', 'SF', 'GIDP']

Pitching columns:
['playerID', 'yearID', 'stint', 'teamID', 'lgID', 'W', 'L', 'G', 'GS', 'CG', 'SHO', 'SV', 'IPouts', 'H', 'ER', 'HR', 'BB', 'SO', 'BAOpp', 'ERA', 'IBB', 'WP', 'HBP', 'BK', 'BFP', 'GF', 'R', 'SH', 'SF', 'GIDP']

Salaries columns:
['yearID', 'teamID', 'lgID', 'playerID', 'salary']

People columns:
['ID', 'playerID', 'birthYear', 'birthMonth', 'birthDay', 'birthCity', 'birthCountry', 'birthState', 'deathYear', 'deathMonth', 'deathDay', 'deathCountry', 'deathState', 'deathCity', 'nameFirst', 'nameLast', 'nameGiven', 'weight', 'height', 'bats', 'throws', 'debut', 'bbrefID', 'finalGame', 'retroID']


In [26]:
def quick_summary(df, name):
    print(f"\n{name}")
    print("-" * len(name))
    print("Shape:", df.shape)
    print("\nColumn names:")
    print(df.columns.tolist())
    print("\nMissing values:")
    print(df.isna().sum())

quick_summary(batting, "Batting")
quick_summary(pitching, "Pitching")
quick_summary(salaries, "Salaries")
quick_summary(people, "People")


Batting
-------
Shape: (128598, 22)

Column names:
['playerID', 'yearID', 'stint', 'teamID', 'lgID', 'G', 'AB', 'R', 'H', '2B', '3B', 'HR', 'RBI', 'SB', 'CS', 'BB', 'SO', 'IBB', 'HBP', 'SH', 'SF', 'GIDP']

Missing values:
playerID        0
yearID          0
stint           0
teamID          0
lgID          737
G               0
AB              0
R               0
H               0
2B              0
3B              0
HR              0
RBI           756
SB           2546
CS          32495
BB              0
SO           8859
IBB         46527
HBP          2816
SH           6068
SF          46664
GIDP        35432
dtype: int64

Pitching
--------
Shape: (57630, 30)

Column names:
['playerID', 'yearID', 'stint', 'teamID', 'lgID', 'W', 'L', 'G', 'GS', 'CG', 'SHO', 'SV', 'IPouts', 'H', 'ER', 'HR', 'BB', 'SO', 'BAOpp', 'ERA', 'IBB', 'WP', 'HBP', 'BK', 'BFP', 'GF', 'R', 'SH', 'SF', 'GIDP']

Missing values:
playerID        0
yearID          0
stint           0
teamID          0
lgID          132

In [27]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf


# LOAD RAW DATA
batting = pd.read_csv(f"{BASE_PATH}/Batting.csv")
salaries = pd.read_csv(f"{BASE_PATH}/Salaries.csv")
people = pd.read_csv(f"{BASE_PATH}/People.csv")


# BUILD SEASON-LEVEL HITTER DATA
bat = batting.copy() #makes copy of original dataset so we do not change in the raw data by mistake

sum_cols = [
    "G","AB","R","H","2B","3B","HR","RBI","SB","CS",
    "BB","SO","IBB","HBP","SH","SF","GIDP"
] #list of batting cols we want to keep. These are season totals like G= games, AB= at bats, H=hits, HR= home runs, BB= walks on


for c in sum_cols:
    if c not in bat.columns:
        bat[c] = 0  #This is a safety step and it says if any expected column is missing in dataset create it and fill it with 0 so the code does not break later

bat_season = (
    bat.groupby(["playerID","yearID"], as_index=False)[sum_cols]
       .sum()
)  #This is the main step that groups the data by playerid and yearid and then adds up all the batting stats in sum_cols

# We do it because one player can appear in multiple rows in the same year, usually if the played for different teams in the same season and multiple stints.
# But for our study we want one row per player per season so this code combines all those rows into one total season record


# Total bases
bat_season["TB"] = (
    bat_season["H"] +
    bat_season["2B"] +
    2 * bat_season["3B"] +
    3 * bat_season["HR"]
)

# OBP
den_obp = bat_season["AB"] + bat_season["BB"] + bat_season["HBP"] + bat_season["SF"]
num_obp = bat_season["H"] + bat_season["BB"] + bat_season["HBP"]

bat_season["OBP"] = np.where(den_obp > 0, num_obp / den_obp, np.nan)

# SLG
bat_season["SLG"] = np.where(
    bat_season["AB"] > 0,
    bat_season["TB"] / bat_season["AB"],
    np.nan
)

# OPS
bat_season["OPS"] = bat_season["OBP"] + bat_season["SLG"]
'''
TB = Total Bases
OBP = On-Base Percentage
SLG = Slugging Percentage
OPS = On-Base Percentage + Slugging Percentage

Then it removes bad or missing values.

OPS is the main batting performance number we use later to measure consistency.

'''

bat_season = bat_season.replace([np.inf, -np.inf], np.nan) #cleans the data by replacing pos and neg infinity values by 0
bat_season = bat_season.dropna(subset=["OPS"]).copy() #drops rows where OPS is missing


MIN_AB = 200 #A hitter season is kept only if the player had 200 at bats to avoidy nosise
MIN_SEASONS = 3 #A player must have 3 qualifying seasons for consistency

qual = bat_season[bat_season["AB"] >= MIN_AB].copy()

season_counts = (
    qual.groupby("playerID")["yearID"]
        .nunique()
        .reset_index(name="n_seasons")
)

qual = qual.merge(season_counts, on="playerID", how="left")
qual = qual[qual["n_seasons"] >= MIN_SEASONS].copy() #cleaned hitter dataset


# BUILD DV
sal_py = (
    salaries.groupby(["playerID","yearID"], as_index=False)["salary"]
    .sum()
    .rename(columns={"salary":"salary_total"})
) # this created 1 total salary record per player, we sum it if salary is split across rows we want the players total salary for that season

sal_py = sal_py.sort_values(["playerID","yearID"]).copy() #calculates salary growth
sal_py["log_salary"] = np.log(sal_py["salary_total"].clip(lower=1)) #takes log of salary cause salaries vary alot log will make it easy to compare
sal_py["prev_log_salary"] = sal_py.groupby("playerID")["log_salary"].shift(1) #players previous year salary
sal_py["dlog_salary"] = sal_py["log_salary"] - sal_py["prev_log_salary"] #this is the change between previous and log salary

cut = sal_py.groupby("yearID")["dlog_salary"].transform(lambda x: x.quantile(0.80)) #this creates a bonus indicator for wheter the player had a top 20% salary jump in that year 1 if jump or else 0
sal_py["promo_jump_top20"] = (sal_py["dlog_salary"] >= cut).astype(int)

#This part creates the salary outcome variable. We first calculate salary growth then shift it forward so each player season is linked to the next observed salary change

# next observed salary-growth outcome
sal_py["dlog_salary_next"] = sal_py.groupby("playerID")["dlog_salary"].shift(-1) #this is players next year observed salary growth. This is the main DV
sal_py["promo_jump_top20_next"] = sal_py.groupby("playerID")["promo_jump_top20"].shift(-1)

dv_sal = sal_py[["playerID","yearID","dlog_salary_next","promo_jump_top20_next"]].copy()


# ROLLING WEIGHTED CV UP TO YEAR t
def rolling_weighted_cv(df, value_col, weight_col, id_col="playerID", time_col="yearID", min_periods=3):
    df = df[[id_col, time_col, value_col, weight_col]].dropna().copy()
    df = df.sort_values([id_col, time_col])

    out_rows = []
    for pid, g in df.groupby(id_col):
        vals = g[value_col].values
        wts = g[weight_col].values
        yrs = g[time_col].values

        for i in range(len(g)):
            v = vals[:i+1]
            w = wts[:i+1]
            y = yrs[i]
            n = len(v)

            if n < min_periods or w.sum() == 0:
                out_rows.append([pid, y, n, np.nan, np.nan, np.nan])
                continue

            wmean = (v * w).sum() / w.sum()
            wsd = np.sqrt((w * (v - wmean) ** 2).sum() / w.sum())
            wcv = wsd / wmean if wmean != 0 else np.nan

            out_rows.append([pid, y, n, wmean, wsd, wcv])

    out = pd.DataFrame(
        out_rows,
        columns=[id_col, time_col, "n_seasons_to_t", "w_mean_to_t", "w_sd_to_t", "w_cv_to_t"]
    )
    return out

'''
What this function does

This is the heart of the consistency measure.

For each player, season by season, it looks at all qualifying seasons up to that year and calculates:
	•	weighted mean performance
	•	weighted standard deviation
	•	weighted coefficient of variation

For hitters:
	•	value_col = "OPS"
	•	weight_col = "AB"

So for each hitter-year, it asks:

looking at all prior qualifying seasons up to now, how consistent has this player’s OPS been?

Why weighted?

Because a season with more at-bats is more informative than a season with fewer at-bats.

Why coefficient of variation?

Because it measures variability relative to the player’s own average level.
	•	low CV = more steady
	•	high CV = more inconsistent

Simple explanation

This function calculates each player’s performance inconsistency over time. It uses all qualifying seasons up to a given year and gives more weight to seasons with more at-bats.
'''

hit_roll = rolling_weighted_cv(
    qual,
    value_col="OPS",
    weight_col="AB",
    id_col="playerID",
    time_col="yearID",
    min_periods=3
).rename(columns={
    "w_mean_to_t":"w_mean_OPS_to_t",
    "w_sd_to_t":"w_sd_OPS_to_t",
    "w_cv_to_t":"w_cv_OPS_to_t"
})

'''
What this does

This applies the rolling consistency function to hitters.

So now hit_roll contains, for each hitter-season:
	•	weighted mean OPS up to that year
	•	weighted SD of OPS up to that year
	•	weighted CV of OPS up to that year

The most important variable is:

w_cv_OPS_to_t

This is your main hitter inconsistency measure.

Simple explanation

This part creates the hitter inconsistency variable using OPS and weighting by at-bats.
'''
# MERGE IV + DV
hit = dv_sal.merge(hit_roll, on=["playerID","yearID"], how="inner") #This combines future salary outcome and the current consistency measure, so now each row has a player a year their consisteancy upto the year their next observed salary growth. This is what we need for regression
hit = hit.dropna(subset=["dlog_salary_next","w_cv_OPS_to_t"]).copy()

# standardize IV
hit["IV_z"] = (
    hit["w_cv_OPS_to_t"] - hit["w_cv_OPS_to_t"].mean()
) / hit["w_cv_OPS_to_t"].std()  #standardizes the inconsisteancy variable that means it rescales it into standard deviation units because then the coefficient is easier to read a 1 standard deviation increase in inconsisteancy changes salary growth by X

hit["IV_z_sq"] = hit["IV_z"] ** 2 #this is just squared to check if the relationship is curved rather than purely straight line

print("Final hitters sample:", hit.shape)
display(hit.head())


# REGRESSION
#does hitter inconsistency predict next observed salary growth?
#DV= dlog_salary_next
#IV= IV_z
#cov_type= HC3, This uses robust standard errors which makes inference safer if variance is not perfectly constant
model_hit = smf.ols(
    "dlog_salary_next ~ IV_z",
    data=hit
).fit(cov_type="HC3")

print(model_hit.summary())

# optional quadratic
# To check if relationship is non linear
#cause maybe inconsistency  is not simply more consistency = worst outcome in a perfectly straight line
#this tests if the pattern curves
model_hit_quad = smf.ols(
    "dlog_salary_next ~ IV_z + IV_z_sq",
    data=hit
).fit(cov_type="HC3")

print(model_hit_quad.summary())

Final hitters sample: (6364, 10)


,playerID,yearID,dlog_salary_next,promo_jump_top20_next,n_seasons_to_t,w_mean_OPS_to_t,w_sd_OPS_to_t,w_cv_OPS_to_t,IV_z,IV_z_sq
2,abbotku01,1996,0.955511,1.0,3,0.732739,0.035561,0.048532,-1.063621,1.131289
3,abbotku01,1997,0.430783,0.0,4,0.735455,0.032527,0.044227,-1.213341,1.472196
4,abbotku01,1999,-0.587787,0.0,5,0.736309,0.029580,0.040173,-1.354319,1.834180
7,abreubo01,2000,0.529893,0.0,3,0.958662,0.036870,0.038460,-1.413914,1.999154
8,abreubo01,2001,0.239795,0.0,4,0.952620,0.033132,0.034780,-1.541887,2.377415


                            OLS Regression Results                            
Dep. Variable:       dlog_salary_next   R-squared:                       0.015
Model:                            OLS   Adj. R-squared:                  0.015
Method:                 Least Squares   F-statistic:                     84.09
Date:                Mon, 20 Apr 2026   Prob (F-statistic):           6.28e-20
Time:                        00:22:13   Log-Likelihood:                -6378.1
No. Observations:                6364   AIC:                         1.276e+04
Df Residuals:                    6362   BIC:                         1.277e+04
Df Model:                           1                                         
Covariance Type:                  HC3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.2354      0.008     28.480      0.0

IV_z = standardized inconsistency

Higher IV_z → more inconsistent player

Lower IV_z → more consistent player

**Conclusion:**
The negative coefficient on IV_z indicates that greater performance inconsistency is associated with lower future salary growth, suggesting that teams value consistent performance when determining player compensation.


In [28]:
# Read hitters result
beta = model_hit.params["IV_z"]
pval = model_hit.pvalues["IV_z"]
'''
This takes the main regression output and pulls out the two most important numbers:
	•	beta = coefficient on IV_z
	•	pval = p-value of IV_z

What do they mean?

beta
This tells us the direction of the relationship.
	•	negative beta → more inconsistency is linked to lower salary growth
	•	positive beta → more inconsistency is linked to higher salary growth

So for our hypothesis, we want beta to be negative.

pval
This tells us whether the result is statistically strong.
	•	p < 0.05 → significant
	•	p >= 0.05 → not significant
'''
print("Hitters coefficient on IV_z:", beta)
print("Hitters p-value:", pval)

if beta < 0 and pval < 0.05:
    print("Result: Supports the hypothesis.")
    print("Interpretation: More inconsistency in hitter performance is associated with lower next-year salary growth.")
elif beta < 0 and pval >= 0.05:
    print("Result: Direction supports the hypothesis, but it is not statistically significant.")
    print("Interpretation: The relationship is negative, but we do not have strong enough evidence.")
elif beta > 0 and pval < 0.05:
    print("Result: Significant, but does NOT support the hypothesis.")
    print("Interpretation: More inconsistency in hitter performance is associated with higher next-year salary growth.")
else:
    print("Result: Does NOT support the hypothesis and is not statistically significant.")
    print("Interpretation: The relationship is positive and weak.")

Hitters coefficient on IV_z: -0.08039878944150354
Hitters p-value: 4.735998233869513e-20
Result: Supports the hypothesis.
Interpretation: More inconsistency in hitter performance is associated with lower next-year salary growth.


In [29]:
# HITTERS: attach primary team to final hitter file

bat_team = batting[["playerID", "yearID", "teamID", "AB"]].copy()
bat_team["AB"] = bat_team["AB"].fillna(0) #This creates a smaller batting table with only the columns needed to identify a payers team in each season

bat_team = bat_team.sort_values(
    ["playerID", "yearID", "AB"],
    ascending=[True, True, False]
) # sorts with highes At Bats to lowest to define primary team

bat_team_primary = (
    bat_team.drop_duplicates(subset=["playerID", "yearID"])
    [["playerID", "yearID", "teamID"]]
    .rename(columns={"teamID": "teamID_primary"})
)

hit_team = hit.merge(
    bat_team_primary,
    on=["playerID", "yearID"],
    how="left"
) # This merges primary team info into final hitter analysis so hit_team is now hitter regression dataset plus a team column.
#This also allows to later run: team summary, fixed effects

print("Hitters file with team column ready")
print(hit_team[["playerID", "yearID", "teamID_primary"]].head())

Hitters file with team column ready
    playerID  yearID teamID_primary
0  abbotku01    1996            FLO
1  abbotku01    1997            FLO
2  abbotku01    1999            COL
3  abreubo01    2000            PHI
4  abreubo01    2001            PHI


**Hitters with team fixed effects**

In [30]:
model_hit_yearFE = smf.ols(
    "dlog_salary_next ~ IV_z + C(yearID)",
    data=hit
).fit(cov_type="HC3")
#extra control(C)=yearID
print(model_hit_yearFE.summary())

'''
Because salaries may change across years due to things like:
	•	inflation
	•	league-wide market changes
	•	contract trends
	•	rule changes
	•	changes in overall team spending

So this model asks:

Does inconsistency still matter even after controlling for differences across years?

Simple explanation

This model adds year fixed effects to control for overall differences across seasons. It checks whether performance inconsistency still predicts salary growth after accounting for year-to-year changes in the salary environment.
'''

                            OLS Regression Results                            
Dep. Variable:       dlog_salary_next   R-squared:                       0.030
Model:                            OLS   Adj. R-squared:                  0.025
Method:                 Least Squares   F-statistic:                     6.966
Date:                Mon, 20 Apr 2026   Prob (F-statistic):           5.20e-29
Time:                        00:22:14   Log-Likelihood:                -6328.3
No. Observations:                6364   AIC:                         1.272e+04
Df Residuals:                    6332   BIC:                         1.294e+04
Df Model:                          31                                         
Covariance Type:                  HC3                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept             0.1567      0.02

'\nBecause salaries may change across years due to things like:\n\t•\tinflation\n\t•\tleague-wide market changes\n\t•\tcontract trends\n\t•\trule changes\n\t•\tchanges in overall team spending\n\nSo this model asks:\n\nDoes inconsistency still matter even after controlling for differences across years?\n\nSimple explanation\n\nThis model adds year fixed effects to control for overall differences across seasons. It checks whether performance inconsistency still predicts salary growth after accounting for year-to-year changes in the salary environment.\n'

**Hitters with year + team fixed effects**

In [31]:
# HITTERS: year + team fixed effects ready file
hit_team_fe = hit_team.dropna(subset=["teamID_primary", "yearID", "IV_z", "dlog_salary_next"]).copy()
#summary check of how many rows,teams and years are in sample
print("Hitters FE sample shape:", hit_team_fe.shape)
print("Unique hitter teams:", hit_team_fe["teamID_primary"].nunique())
print("Unique hitter years:", hit_team_fe["yearID"].nunique())

Hitters FE sample shape: (6364, 11)
Unique hitter teams: 35
Unique hitter years: 31


In [32]:
# HITTERS: year + team fixed effects model
model_hit_year_teamFE = smf.ols(
    "dlog_salary_next ~ IV_z + C(yearID) + C(teamID_primary)",
    data=hit_team_fe
).fit(cov_type="HC3")

print(model_hit_year_teamFE.summary())

                            OLS Regression Results                            
Dep. Variable:       dlog_salary_next   R-squared:                       0.045
Model:                            OLS   Adj. R-squared:                  0.035
Method:                 Least Squares   F-statistic:                     4.666
Date:                Mon, 20 Apr 2026   Prob (F-statistic):           1.71e-31
Time:                        00:22:14   Log-Likelihood:                -6277.9
No. Observations:                6364   AIC:                         1.269e+04
Df Residuals:                    6298   BIC:                         1.313e+04
Df Model:                          65                                         
Covariance Type:                  HC3                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

In [33]:
print(model_hit_yearFE.params["IV_z"])
print(model_hit_yearFE.pvalues["IV_z"])
print(model_hit_year_teamFE.params["IV_z"])
print(model_hit_year_teamFE.pvalues["IV_z"])

-0.08001911762095917
1.7351860560854485e-19
-0.07763794467977377
2.695457220144332e-18


**How to read?**

With year fixed effects, the hitter inconsistency coefficient is still negative and highly significant. With both year(salary env) and team fixed effects, it remains negative and highly significant. So the hypothesis still holds even after controlling for season-level and team-level differences.

**Interpretation Conclusion:**

More inconsistency in performance predicts lower subsequent salary growth, even after accounting for differences across years and teams.

# **Pitchers**

In [34]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf


# LOAD RAW DATA
pitching = pd.read_csv(f"{BASE_PATH}/Pitching.csv")
salaries = pd.read_csv(f"{BASE_PATH}/Salaries.csv")
people = pd.read_csv(f"{BASE_PATH}/People.csv")


# BUILD SEASON-LEVEL PITCHER DATA
pit = pitching.copy()  # makes a copy so we do not accidentally change the raw dataset

sum_cols = [
    "W", "L", "G", "GS", "CG", "SHO", "SV",
    "IPouts", "H", "ER", "HR", "BB", "SO",
    "BAOpp", "ERA", "IBB", "WP", "HBP", "BK",
    "BFP", "GF", "R", "SH", "SF", "GIDP"
]  # list of pitching stats we want to keep (season totals)

for c in sum_cols:
    if c not in pit.columns:
        pit[c] = 0  # safety step: if any column is missing, create it with 0 so code does not break

pit_season = (
    pit.groupby(["playerID", "yearID"], as_index=False)[sum_cols]
       .sum()
)
# groups data by player and year, then sums stats
# needed because pitchers can have multiple rows in one season (multiple teams)
# this gives us ONE row per pitcher per season


# innings pitched
pit_season["IP"] = pit_season["IPouts"] / 3
# converts outs into innings (3 outs = 1 inning)

# ERA
pit_season["ERA"] = np.where(
    pit_season["IP"] > 0,
    9 * pit_season["ER"] / pit_season["IP"],
    np.nan
)
'''
ERA = Earned Run Average

Formula:
ERA = (Earned Runs / Innings Pitched) * 9

This standardizes runs allowed per 9 innings.

ERA is the main pitching performance metric we use later to measure consistency.
'''

pit_season = pit_season.replace([np.inf, -np.inf], np.nan)  # clean infinities
pit_season = pit_season.dropna(subset=["ERA"]).copy()  # remove missing ERA


# QUALIFICATION FILTERS
MIN_IP = 50  # only keep seasons with at least 50 innings pitched
MIN_SEASONS = 3  # pitcher must have at least 3 qualifying seasons

qual = pit_season[pit_season["IP"] >= MIN_IP].copy()

season_counts = (
    qual.groupby("playerID")["yearID"]
        .nunique()
        .reset_index(name="n_seasons")
)

qual = qual.merge(season_counts, on="playerID", how="left")
qual = qual[qual["n_seasons"] >= MIN_SEASONS].copy()
# final cleaned pitcher dataset


# BUILD DV SAME WAY AS HITTERS

sal_py = (
    salaries.groupby(["playerID", "yearID"], as_index=False)["salary"]
    .sum()
    .rename(columns={"salary": "salary_total"})
)
# creates total salary per player per year

sal_py = sal_py.sort_values(["playerID", "yearID"]).copy()

sal_py["log_salary"] = np.log(sal_py["salary_total"].clip(lower=1))
# take log of salary to stabilize large differences

sal_py["prev_log_salary"] = sal_py.groupby("playerID")["log_salary"].shift(1)
# previous year's salary

sal_py["dlog_salary"] = sal_py["log_salary"] - sal_py["prev_log_salary"]
# salary growth (log difference)

cut = sal_py.groupby("yearID")["dlog_salary"].transform(lambda x: x.quantile(0.80))
sal_py["promo_jump_top20"] = (sal_py["dlog_salary"] >= cut).astype(int)
# indicator for top 20% salary jump

# next observed salary-growth outcome
sal_py["dlog_salary_next"] = sal_py.groupby("playerID")["dlog_salary"].shift(-1)
sal_py["promo_jump_top20_next"] = sal_py.groupby("playerID")["promo_jump_top20"].shift(-1)

dv_sal = sal_py[["playerID", "yearID", "dlog_salary_next", "promo_jump_top20_next"]].copy()

'''
This part creates the salary outcome variable.

We calculate salary growth and then shift it forward so each pitcher-season
is linked to the NEXT observed salary change.

So:
performance today → salary change next period
'''


# ROLLING WEIGHTED CV UP TO YEAR t
def rolling_weighted_cv(df, value_col, weight_col, id_col="playerID", time_col="yearID", min_periods=3):
    df = df[[id_col, time_col, value_col, weight_col]].dropna().copy()
    df = df.sort_values([id_col, time_col])

    out_rows = []
    for pid, g in df.groupby(id_col):
        vals = g[value_col].values
        wts = g[weight_col].values
        yrs = g[time_col].values

        for i in range(len(g)):
            v = vals[:i+1]
            w = wts[:i+1]
            y = yrs[i]
            n = len(v)

            if n < min_periods or w.sum() == 0:
                out_rows.append([pid, y, n, np.nan, np.nan, np.nan])
                continue

            wmean = (v * w).sum() / w.sum()
            wsd = np.sqrt((w * (v - wmean) ** 2).sum() / w.sum())
            wcv = wsd / wmean if wmean != 0 else np.nan

            out_rows.append([pid, y, n, wmean, wsd, wcv])

    out = pd.DataFrame(
        out_rows,
        columns=[id_col, time_col, "n_seasons_to_t", "w_mean_to_t", "w_sd_to_t", "w_cv_to_t"]
    )
    return out

'''
What this function does

This is the core consistency measure.

For each pitcher, year by year, it looks at all prior qualifying seasons
and calculates:
	•	weighted mean ERA
	•	weighted standard deviation
	•	weighted coefficient of variation (CV)

For pitchers:
	•	value_col = "ERA"
	•	weight_col = "IP"

So it asks:

Looking at all seasons up to now, how consistent has this pitcher’s ERA been?

Why weighted?

Because a season with more innings pitched is more reliable than one with fewer innings.

Why coefficient of variation?

Because it measures variability relative to the pitcher’s own average.
	•	low CV = consistent
	•	high CV = inconsistent

Simple explanation

This function calculates pitcher inconsistency over time,
giving more weight to seasons with more innings pitched.
'''


pit_roll = rolling_weighted_cv(
    qual,
    value_col="ERA",
    weight_col="IP",
    id_col="playerID",
    time_col="yearID",
    min_periods=3
).rename(columns={
    "w_mean_to_t": "w_mean_ERA_to_t",
    "w_sd_to_t": "w_sd_ERA_to_t",
    "w_cv_to_t": "w_cv_ERA_to_t"
})

'''
What this does

This applies the rolling consistency function to pitchers.

Now pit_roll contains, for each pitcher-season:
	•	weighted mean ERA up to that year
	•	weighted SD of ERA
	•	weighted CV of ERA

The key variable is:

w_cv_ERA_to_t

This is your pitcher inconsistency measure.

Simple explanation

This creates the pitcher inconsistency variable using ERA,
weighted by innings pitched.
'''


# MERGE IV + DV
pit_final = dv_sal.merge(pit_roll, on=["playerID", "yearID"], how="inner")
# combines future salary outcome with current consistency measure

pit_final = pit_final.dropna(subset=["dlog_salary_next", "w_cv_ERA_to_t"]).copy()

# standardize IV
pit_final["IV_z"] = (
    pit_final["w_cv_ERA_to_t"] - pit_final["w_cv_ERA_to_t"].mean()
) / pit_final["w_cv_ERA_to_t"].std()
# standardizes inconsistency (in SD units)

pit_final["IV_z_sq"] = pit_final["IV_z"] ** 2
# squared term to test for nonlinear relationship


# REGRESSION
# does pitcher inconsistency predict next observed salary growth?

model_pit = smf.ols(
    "dlog_salary_next ~ IV_z",
    data=pit_final
).fit(cov_type="HC3")

print(model_pit.summary())
'''
DV = dlog_salary_next (next-period salary growth)
IV = IV_z (pitcher inconsistency)

HC3 = robust standard errors

This model tests:

Does more inconsistency in ERA affect future salary growth?
'''

                            OLS Regression Results                            
Dep. Variable:       dlog_salary_next   R-squared:                       0.005
Model:                            OLS   Adj. R-squared:                  0.005
Method:                 Least Squares   F-statistic:                     26.15
Date:                Mon, 20 Apr 2026   Prob (F-statistic):           3.26e-07
Time:                        00:22:16   Log-Likelihood:                -5557.0
No. Observations:                5283   AIC:                         1.112e+04
Df Residuals:                    5281   BIC:                         1.113e+04
Df Model:                           1                                         
Covariance Type:                  HC3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.3108      0.010     32.600      0.0

'\nDV = dlog_salary_next (next-period salary growth)\nIV = IV_z (pitcher inconsistency)\n\nHC3 = robust standard errors\n\nThis model tests:\n\nDoes more inconsistency in ERA affect future salary growth?\n'

In [35]:
# PITCHERS: attach primary team to final pitcher file

pit_team = pitching[["playerID", "yearID", "teamID", "IPouts"]].copy()
# makes a smaller pitcher table with only the columns needed to identify the main team

pit_team["IPouts"] = pit_team["IPouts"].fillna(0)
# fills missing IPouts with 0 so sorting and selection work correctly

pit_team = pit_team.sort_values(
    ["playerID", "yearID", "IPouts"],
    ascending=[True, True, False]
)
# sorts by player and year, then puts the team with the most outs first
# this helps us pick the main team for each pitcher-season

pit_team_primary = (
    pit_team.drop_duplicates(subset=["playerID", "yearID"])
    [["playerID", "yearID", "teamID"]]
    .rename(columns={"teamID": "teamID_primary"})
)
# keeps only one row per player-season
# renames teamID to teamID_primary because this is the pitcher’s main team

pit_team_final = pit_final.merge(
    pit_team_primary,
    on=["playerID", "yearID"],
    how="left"
)
# merges the primary team info into the final pitcher analysis dataset

print("Pitchers file with team column ready")
print(pit_team_final[["playerID", "yearID", "teamID_primary"]].head())
# quick check to confirm the team column was added correctly

Pitchers file with team column ready
    playerID  yearID teamID_primary
0   aasedo01    1986            BAL
1  abbotji01    1991            CAL
2  abbotji01    1992            CAL
3  abbotji01    1993            NYA
4  abbotji01    1994            NYA


**Pitchers with year fixed effects**

In [36]:
model_pit_yearFE = smf.ols(
   "dlog_salary_next ~ IV_z + C(yearID)",
   data=pit_final
).fit(cov_type="HC3")

print(model_pit_yearFE.summary())

'''
This model adds year fixed effects for pitchers.

Why add C(yearID)?

Because salaries can change over time due to:
	•	inflation
	•	league-wide salary growth
	•	changes in contracts
	•	rule changes
	•	team spending trends

So we want to control for differences across years.

What this model tests

Does pitcher inconsistency (IV_z) still affect salary growth
AFTER controlling for differences between years?

Simple explanation

This model checks if inconsistency still matters when we account
for the fact that different years have different salary environments.
'''


                            OLS Regression Results                            
Dep. Variable:       dlog_salary_next   R-squared:                       0.029
Model:                            OLS   Adj. R-squared:                  0.023
Method:                 Least Squares   F-statistic:                     5.461
Date:                Mon, 20 Apr 2026   Prob (F-statistic):           1.39e-20
Time:                        00:22:16   Log-Likelihood:                -5493.1
No. Observations:                5283   AIC:                         1.105e+04
Df Residuals:                    5251   BIC:                         1.126e+04
Df Model:                          31                                         
Covariance Type:                  HC3                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept             0.1739      0.04

'\nThis model adds year fixed effects for pitchers.\n\nWhy add C(yearID)?\n\nBecause salaries can change over time due to:\n\t•\tinflation\n\t•\tleague-wide salary growth\n\t•\tchanges in contracts\n\t•\trule changes\n\t•\tteam spending trends\n\nSo we want to control for differences across years.\n\nWhat this model tests\n\nDoes pitcher inconsistency (IV_z) still affect salary growth\nAFTER controlling for differences between years?\n\nSimple explanation\n\nThis model checks if inconsistency still matters when we account\nfor the fact that different years have different salary environments.\n'

**Pitchers with year + team fixed effects**

In [37]:
# PITCHERS: year + team fixed effects ready file
pit_team_fe = pit_team_final.dropna(subset=["teamID_primary", "yearID", "IV_z", "dlog_salary_next"]).copy()

print("Pitchers FE sample shape:", pit_team_fe.shape)
print("Unique pitcher teams:", pit_team_fe["teamID_primary"].nunique())
print("Unique pitcher years:", pit_team_fe["yearID"].nunique())
'''
This step prepares the dataset for the full fixed effects model.

What it does

It removes any rows with missing:
	•	teamID_primary (team)
	•	yearID (year)
	•	IV_z (inconsistency measure)
	•	dlog_salary_next (future salary growth)

Why this matters

We need complete data for:
	•	team fixed effects
	•	year fixed effects
	•	regression variables

The print statements are just checks to see:
	•	how many observations we have
	•	how many teams are included
	•	how many years are included

Simple explanation

This step cleans the pitcher dataset so it is ready
for the full regression with team and year controls.
'''

Pitchers FE sample shape: (5283, 11)
Unique pitcher teams: 35
Unique pitcher years: 31


'\nThis step prepares the dataset for the full fixed effects model.\n\nWhat it does\n\nIt removes any rows with missing:\n\t•\tteamID_primary (team)\n\t•\tyearID (year)\n\t•\tIV_z (inconsistency measure)\n\t•\tdlog_salary_next (future salary growth)\n\nWhy this matters\n\nWe need complete data for:\n\t•\tteam fixed effects\n\t•\tyear fixed effects\n\t•\tregression variables\n\nThe print statements are just checks to see:\n\t•\thow many observations we have\n\t•\thow many teams are included\n\t•\thow many years are included\n\nSimple explanation\n\nThis step cleans the pitcher dataset so it is ready\nfor the full regression with team and year controls.\n'

In [38]:
# PITCHERS: year + team fixed effects model
model_pit_year_teamFE = smf.ols(
    "dlog_salary_next ~ IV_z + C(yearID) + C(teamID_primary)",
    data=pit_team_fe
).fit(cov_type="HC3")

print(model_pit_year_teamFE.summary())
'''
This is the full model with both year and team fixed effects.

What is added here?

In addition to year effects, we now control for team differences.

Why control for teams?

Different teams may:
	•	pay differently
	•	value consistency differently
	•	have different budgets
	•	use different evaluation strategies

What this model tests

Does pitcher inconsistency affect salary growth
AFTER controlling for:
	•	time differences (years)
	•	team differences

Simple explanation

This model checks if inconsistency matters even when comparing
pitchers within the same team and same year environment.
'''

                            OLS Regression Results                            
Dep. Variable:       dlog_salary_next   R-squared:                       0.046
Model:                            OLS   Adj. R-squared:                  0.034
Method:                 Least Squares   F-statistic:                     4.259
Date:                Mon, 20 Apr 2026   Prob (F-statistic):           4.96e-27
Time:                        00:22:16   Log-Likelihood:                -5445.8
No. Observations:                5283   AIC:                         1.102e+04
Df Residuals:                    5217   BIC:                         1.146e+04
Df Model:                          65                                         
Covariance Type:                  HC3                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

'\nThis is the full model with both year and team fixed effects.\n\nWhat is added here?\n\nIn addition to year effects, we now control for team differences.\n\nWhy control for teams?\n\nDifferent teams may:\n\t•\tpay differently\n\t•\tvalue consistency differently\n\t•\thave different budgets\n\t•\tuse different evaluation strategies\n\nWhat this model tests\n\nDoes pitcher inconsistency affect salary growth\nAFTER controlling for:\n\t•\ttime differences (years)\n\t•\tteam differences\n\nSimple explanation\n\nThis model checks if inconsistency matters even when comparing\npitchers within the same team and same year environment.\n'

**How to read?**

In [39]:
print(model_pit_yearFE.params["IV_z"])
print(model_pit_yearFE.pvalues["IV_z"])
print(model_pit_year_teamFE.params["IV_z"])
print(model_pit_year_teamFE.pvalues["IV_z"])
'''
This extracts the key results from the pitcher models.

What it pulls out

For both models:
	•	coefficient (beta) on IV_z
	•	p-value of IV_z

What do these mean?

beta
	•	shows direction of relationship
	•	negative → more inconsistency → lower salary growth
	•	positive → more inconsistency → higher salary growth

p-value
	•	tells us if the result is statistically significant
	•	small p-value (usually < 0.05) → strong evidence
	•	large p-value → weak or no evidence

Why we check both models

We compare:
	•	model with only year effects
	•	model with year + team effects

to see if results are consistent.

Simple explanation

This step shows whether pitcher inconsistency actually matters,
and whether the result is statistically reliable.
'''

-0.052384102393822664
4.005981562065481e-08
-0.0488293745294781
3.56166504522469e-07


'\nThis extracts the key results from the pitcher models.\n\nWhat it pulls out\n\nFor both models:\n\t•\tcoefficient (beta) on IV_z\n\t•\tp-value of IV_z\n\nWhat do these mean?\n\nbeta\n\t•\tshows direction of relationship\n\t•\tnegative → more inconsistency → lower salary growth\n\t•\tpositive → more inconsistency → higher salary growth\n\np-value\n\t•\ttells us if the result is statistically significant\n\t•\tsmall p-value (usually < 0.05) → strong evidence\n\t•\tlarge p-value → weak or no evidence\n\nWhy we check both models\n\nWe compare:\n\t•\tmodel with only year effects\n\t•\tmodel with year + team effects\n\nto see if results are consistent.\n\nSimple explanation\n\nThis step shows whether pitcher inconsistency actually matters,\nand whether the result is statistically reliable.\n'

We measure performance steadiness using a rolling weighted coefficient of variation up to season t. For hitters, variability is based on OPS and weighted by at-bats. For pitchers, variability is based on ERA and weighted by innings pitched. We then test whether greater inconsistency predicts lower next observed salary growth. In both samples, the coefficient is negative and statistically significant, supporting the hypothesis.

# **Robustness Check using Awards**

In [40]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# Load the awards data
awards = pd.read_csv(f"{BASE_PATH}/AwardsPlayers.csv")

# Count how many awards each player received in each year
awards_year = (
    awards.groupby(["playerID", "yearID"])
          .size()
          .reset_index(name="award_count")
          .sort_values(["playerID", "yearID"])
)

# Create a running total of awards for each player over time
awards_year["cum_awards"] = awards_year.groupby("playerID")["award_count"].cumsum()

# --- Add award history to hitter data ---
hit_aw = hit.sort_values(["playerID", "yearID"]).copy()

# Merge yearly award counts and cumulative awards into the hitter panel
hit_aw = hit_aw.merge(
    awards_year[["playerID", "yearID", "award_count", "cum_awards"]],
    on=["playerID", "yearID"],
    how="left"
)

# Missing award values mean the player had no award that year
hit_aw["award_count"] = hit_aw["award_count"].fillna(0)

# Carry the most recent cumulative award total forward within each player
hit_aw["cum_awards"] = hit_aw.groupby("playerID")["cum_awards"].ffill().fillna(0)

# Awards earned before the current season
hit_aw["prior_awards"] = hit_aw["cum_awards"] - hit_aw["award_count"]

# Log version of prior awards
hit_aw["log_prior_awards"] = np.log1p(hit_aw["prior_awards"])

# Run the robustness regression for hitters
model_hit_awards = smf.ols(
    "dlog_salary_next ~ IV_z + log_prior_awards",
    data=hit_aw
).fit(cov_type="HC3")

print("HITTERS: Awards robustness")
print(model_hit_awards.summary())

# --- Add award history to pitcher data ---
pit_aw = pit_final.sort_values(["playerID", "yearID"]).copy()

# Merge yearly award counts and cumulative awards into the pitcher panel
pit_aw = pit_aw.merge(
    awards_year[["playerID", "yearID", "award_count", "cum_awards"]],
    on=["playerID", "yearID"],
    how="left"
)

# Missing award values mean the player had no award that year
pit_aw["award_count"] = pit_aw["award_count"].fillna(0)

# Carry the most recent cumulative award total forward within each player
pit_aw["cum_awards"] = pit_aw.groupby("playerID")["cum_awards"].ffill().fillna(0)

# Awards earned before the current season
pit_aw["prior_awards"] = pit_aw["cum_awards"] - pit_aw["award_count"]

# Log version of prior awards
pit_aw["log_prior_awards"] = np.log1p(pit_aw["prior_awards"])

# Run the robustness regression for pitchers
model_pit_awards = smf.ols(
    "dlog_salary_next ~ IV_z + log_prior_awards",
    data=pit_aw
).fit(cov_type="HC3")

print("\nPITCHERS: Awards robustness")
print(model_pit_awards.summary())

HITTERS: Awards robustness
                            OLS Regression Results                            
Dep. Variable:       dlog_salary_next   R-squared:                       0.050
Model:                            OLS   Adj. R-squared:                  0.050
Method:                 Least Squares   F-statistic:                     182.9
Date:                Mon, 20 Apr 2026   Prob (F-statistic):           5.92e-78
Time:                        00:22:17   Log-Likelihood:                -6262.2
No. Observations:                6364   AIC:                         1.253e+04
Df Residuals:                    6361   BIC:                         1.255e+04
Df Model:                           2                                         
Covariance Type:                  HC3                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept    

**Awards Robustness Conclusion**

As an additional robustness check, we control for player recognition by incorporating award history into the analysis. Awards serve as a proxy for player visibility and reputation, which may independently influence salary outcomes.

We construct measures of prior awards using the cumulative number of awards earned before season t. We include both a continuous specification (log-transformed award count) and a binary indicator for whether a player has received any prior awards.

The results remain consistent with the baseline findings. Performance inconsistency continues to have a negative and statistically significant effect on subsequent salary growth for both hitters and pitchers across all specifications. Although the magnitude of the coefficient is somewhat reduced, the effect remains economically and statistically meaningful.

Interestingly, the award variables themselves are negative and statistically significant. This suggests that players with prior awards tend to experience lower subsequent salary growth, likely reflecting a ceiling effect where highly recognized players are already highly compensated and have less room for further increases.

Overall, these results reinforce the main conclusion that performance inconsistency is negatively associated with salary growth, and that this relationship is not driven by differences in player recognition or visibility.

# **Extra Info**

**Hitter / Batter Terminology (from dataset)**

**Basic Counting Stats**

G (Games)
Number of games the player appeared in.

AB (At Bats)
Official batting attempts (excludes walks, sacrifices, etc.).

R (Runs Scored)
Number of times the player scored a run.

H (Hits)
Number of successful hits.

2B (Doubles)
Hits where the batter reaches second base.

3B (Triples)
Hits where the batter reaches third base.

HR (Home Runs)
Hits where the batter scores automatically.

RBI (Runs Batted In)
Number of runs driven in by the batter.

**Plate Discipline / Approach**

BB (Walks)
Number of times the batter reached base via walk.

SO (Strikeouts)
Number of times the batter struck out.

IBB (Intentional Walks)
Walks given intentionally by the opposing team.

HBP (Hit By Pitch)
Times the batter was hit by a pitch.

**Base Running**

SB (Stolen Bases)
Number of successful stolen base attempts.

CS (Caught Stealing)
Number of times the player was thrown out stealing.

**Situational Stats**

SH (Sacrifice Hits / Bunts)
Plays where the batter advances a runner at the cost of an out.

SF (Sacrifice Flies)
Fly balls that result in a run scoring.

GIDP (Grounded Into Double Play)
Number of double plays hit into.

**Key Rate Statistics**

AVG (Batting Average)
Measures hitting success rate.

Formula:

AVG = H / AB

Higher AVG = better hitter

OBP (On-Base Percentage) (if calculated in your code)
Measures how often a batter reaches base.

Formula:

OBP = (H + BB + HBP) / (AB + BB + HBP + SF)

More complete than AVG

SLG (Slugging Percentage) (if calculated)
Measures power by weighting extra-base hits.

Formula:

SLG = Total Bases / AB

OPS (On-base + Slugging) (if calculated)

Combines ability to get on base and hit for power.

Formula:

OPS = OBP + SLG

**Key Variables Used in This Project**

Performance Metric (e.g., AVG or OPS)
Used to measure hitter performance.

AB (Weighting Variable)
Used to weight seasons — more at-bats = more reliable performance.

w_cv_<stat>_to_t (Inconsistency Measure)
Weighted coefficient of variation of the chosen stat over time.

Low value → consistent hitter

High value → inconsistent hitter

**Simple Summary**

AVG / OPS tells us how good a hitter is

AB tells us how much they played (reliability)

w_cv_<stat>_to_t tells us how consistent they are over time


**Pitcher Terminology (from dataset)**

**Basic Counting Stats**

W (Wins)
Number of games where the pitcher was credited with the win.

L (Losses)
Number of games where the pitcher was charged with the loss.

G (Games)
Total number of games the pitcher appeared in.

GS (Games Started)
Number of games the pitcher started.

CG (Complete Games)
Games where the pitcher pitched the entire game.

SHO (Shutouts)
Complete games where the pitcher allowed 0 runs.

SV (Saves)
Games where the pitcher finished a win under save conditions.

**Workload / Playing Time**

IPouts
Number of outs recorded by the pitcher.
(3 outs = 1 inning)

IP (Innings Pitched)
Total innings pitched.
Calculated as: IP = IPouts / 3

BFP (Batters Faced)
Total number of batters the pitcher faced.

**Performance Stats**

H (Hits Allowed)
Number of hits given up by the pitcher.

ER (Earned Runs)
Runs allowed that are not due to fielding errors.

R (Runs Allowed)
Total runs allowed (includes unearned runs).

HR (Home Runs Allowed)
Number of home runs allowed.

BB (Walks)
Number of batters walked.

SO (Strikeouts)
Number of batters struck out.

BAOpp (Batting Average Against)
Opponents’ batting average against the pitcher.

**Key Rate Statistic**

ERA (Earned Run Average)
Measures runs allowed per 9 innings.

Formula:

ERA = (ER / IP) * 9

Lower ERA = better performance

Main performance metric used in this analysis

**Situational / Advanced Context Stats**

IBB (Intentional Walks)
Walks given intentionally.

WP (Wild Pitches)
Pitches that are too difficult for the catcher to handle.

HBP (Hit By Pitch)
Number of batters hit by the pitcher.

BK (Balks)
Illegal pitching motions that allow runners to advance.

GF (Games Finished)
Games where the pitcher was the final pitcher.

SH (Sacrifice Hits Allowed)
Sacrifice bunts allowed.

SF (Sacrifice Flies Allowed)
Sacrifice flies allowed.

GIDP (Grounded Into Double Play)
Double plays induced by the pitcher.

**Key Variable Used in This Project**

ERA (Performance Metric)
Used to measure pitcher effectiveness.

IP (Weighting Variable)
Used to weight seasons — more innings = more reliable performance.

w_cv_ERA_to_t (Inconsistency Measure)
Weighted coefficient of variation of ERA over time.

Low value → consistent pitcher

High value → inconsistent pitcher

**Simple Summary**

ERA tells us how good a pitcher is

IP tells us how much they pitched (reliability)

w_cv_ERA_to_t tells us how consistent they are over time


**Weighted CV:**

Hitters: weighted by At Bats (AB)

Pitchers: weighted by Innings Pitched (IP)

**Why?**

Because seasons with more playing opportunity are more reliable. We don't want skewness instead want accurate

So:

  
  •	a hitter season with 500 at-bats should count more than one with 30 at-bats


  •	a pitcher season with 180 innings pitched should count more than one with 8 innings

**In code above:**


  •	hitters: weight_col="AB"

  
  •	pitchers: weight_col="IP"



We weight hitter consistency by at-bats and pitcher consistency by innings pitched, so seasons with more meaningful playing time have more influence on the consistency measure.

**HC3 Robustness:**

We report heteroskedasticity-robust (HC3) standard errors to ensure that our statistical inference is reliable even if the variability in the error terms is not constant across observations. In our setting, salary changes and player performance can vary widely across individuals, making it likely that the standard OLS assumption of constant variance (homoscedasticity) is violated. While this does not affect the estimated coefficients themselves, it can lead to incorrect standard errors and misleading p-values. The HC3 correction adjusts the standard errors to account for this issue, providing more accurate significance tests. We use HC3 specifically because it is a more conservative and reliable adjustment for real-world data, helping ensure that our results are not driven by uneven variability in the sample.

NameError: name 'bat_final' is not defined